# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedr456/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Note: the research-paper resource linked on this card was still marked "being prepared" on
the portal when I wrote this, so I audit the two headline findings FlyRank has published
publicly (the DTC-brand case study shared July 2026). Constructive tone throughout — these
are "how to make it stronger" questions, the same ones I ask my own numbers below.*

**Finding A — "$941K in subscription revenue tracked back to articles we wrote for one
brand; a 12.23× return."**
*Where does the label come from?* "Tracked back" is an attribution model — last-touch,
first-touch, or assisted? The revenue label is produced by that model, so the 12.23× is a
property of the attribution choice as much as of the articles. *Does the design carry the
claim?* One brand, chosen after the results were visible, with no counterfactual (what
would organic revenue have been without the articles?) supports **association within one
portfolio**, not a general return multiple. What would raise it a rung: state the
attribution window/model, and add a matched comparison (similar brand or pre/post with
seasonality control).

**Finding B — "74.8% of it is recurring subscription revenue."**
*Where does the label come from?* A share needs a cohort window: subscription revenue
accrues over time, so the share depends on when it was measured (right-censoring — newer
cohorts haven't had time to recur). *Does the design carry the claim?* As a description of
this one brand's measured revenue mix, yes; as evidence that "content compounds," it needs
the cohort curve, not one cut. Honest form: "of revenue attributed by [model] through
[date], 74.8% came from recurring subscriptions."

The same standard is now applied to my own model.

In [1]:
# The audit standard, applied to me: my label's origin, stated as data.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/mohamedr456/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
y = (df["trend_direction"].str.lower() == "down").astype(int)
print("My label: trend_direction == 'down' — observed trailing-90d decline, derived from")
print("trend_pct. It describes the SAME window as the features (a within-snapshot ranking")
print("target, not a forecast). Distribution:")
print(df["trend_direction"].str.lower().value_counts().to_string())
print(f"\nbase rate (declining): {y.mean():.3f}")

My label: trend_direction == 'down' — observed trailing-90d decline, derived from
trend_pct. It describes the SAME window as the features (a within-snapshot ranking
target, not a forecast). Distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152

base rate (declining): 0.542


## 2. My model under an honest split (before/after)

Same features, same model, same metric — only the split changes. **Random (stratified)
5-fold vs client-grouped 5-fold:**

| split | P@50 | AUC |
|---|---|---|
| random 5-fold | **0.920** | 0.767 |
| client-holdout 5-fold | **0.700** | 0.653 |

The **0.22 gap is memorization**: under a random split, rows from every client sit on both
sides, and the model partly recognizes clients instead of learning transferable signal.
The honest number for "will this queue work on a client we never saw?" is 0.700 — still
+0.14 over the frozen rule and +0.16 over random ordering, and that's the only number my
paper will headline.

In [2]:
# Before/after, computed live: random vs grouped, same everything else.
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

X = pd.DataFrame({
    "content_age_days": df["content_age_days"],
    "days_since_last_update": df["days_since_last_update"],
    "log_impressions_90d": np.log1p(df["impressions_90d"]),
    "avg_position": df["avg_position"].replace(0, np.nan),
    "has_position": (df["avg_position"] > 0).astype(int),
    "ctr": df["ctr"],
    "word_count": df["word_count"],
    "has_word_count": df["word_count"].notna().astype(int),
    "engagement_rate": df["engagement_rate"],
})
X["avg_position"] = X["avg_position"].fillna(X["avg_position"].median())
X["word_count"] = X["word_count"].fillna(X["word_count"].median())
X["engagement_rate"] = X["engagement_rate"].fillna(0)

def p_at_k(s, l, k=50):
    o = np.argsort(-np.asarray(s)); return float(np.asarray(l)[o[:k]].mean())

def cv_scores(splits, Xf):
    ps, aucs = [], []
    for tr, te in splits:
        rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                    n_jobs=-1, random_state=42).fit(Xf.iloc[tr], y.iloc[tr])
        p = rf.predict_proba(Xf.iloc[te])[:, 1]
        ps.append(p_at_k(p, y.iloc[te])); aucs.append(roc_auc_score(y.iloc[te], p))
    return round(float(np.mean(ps)), 3), round(float(np.mean(aucs)), 3)

grouped = list(GroupKFold(5).split(X, y, groups=df["client_id"]))
random_ = list(StratifiedKFold(5, shuffle=True, random_state=42).split(X, y))
gp, ga = cv_scores(grouped, X)
rp, ra = cv_scores(random_, X)
print(f"random split   : P@50 {rp}  AUC {ra}")
print(f"client-holdout : P@50 {gp}  AUC {ga}")
print(f"memorization gap: P@50 {rp - gp:+.3f}, AUC {ra - ga:+.3f} — the gap IS the finding")

import json as _json
os.makedirs("work/outputs", exist_ok=True)
with open("work/outputs/validation_audit_metrics.json", "w") as f:
    _json.dump({"random_split": {"p50": rp, "auc": ra},
                "client_holdout": {"p50": gp, "auc": ga},
                "leak_injection_grouped": "see next cell", "seed": 42}, f, indent=2)

random split   : P@50 0.92  AUC 0.767
client-holdout : P@50 0.7  AUC 0.653
memorization gap: P@50 +0.220, AUC +0.114 — the gap IS the finding


## 3. Leakage audit

The attack checklist from the leakage skill, on the final feature set:

- **Timeline:** every feature is a trailing-window fact. Named honestly: the label is also
  a trailing-window fact (same 90 days) — so this model **ranks concurrent decline**, it
  does not forecast. The forecasting version needs the warehouse's future-window label
  (rehearsed in `w03_data_contract`, where features Mar 1–28 predict clicks Mar 29–31).
- **Label-derived columns:** `trend_direction`, `trend_pct`, `is_declining_label` are
  banned from features — and the deliberate-injection test below shows why: adding
  `trend_pct` sends P@50 and AUC to a perfect **1.000** (the label is literally computed
  from it). Removed; the honest 0.700 stands. A perfect score is a confession, and the
  test harness passing this check means it can catch real leaks.
- **Product flags:** none exist in the starter release (verified in w04).
- **IDs:** `client_id`/`content_id` are pseudonyms used for grouping only.
- **Base rate:** printed next to every metric in every notebook (0.542 here).

In [3]:
# The deliberate injection: trend_pct (the label's parent) as a 'feature'.
X_leak = X.assign(trend_pct=df["trend_pct"])
lp, la = cv_scores(grouped, X_leak)
print(f"client-holdout + trend_pct injected: P@50 {lp}  AUC {la}")
print(f"clean client-holdout               : P@50 {gp}  AUC {ga}")
print("\nThe jump to perfect is the leak confessing. trend_pct stays banned;")
FORBIDDEN = {"trend_direction", "trend_pct", "is_declining_label", "content_id", "client_id"}
assert not set(X.columns) & FORBIDDEN
print("final feature set verified clean:", sorted(X.columns))

client-holdout + trend_pct injected: P@50 1.0  AUC 1.0
clean client-holdout               : P@50 0.7  AUC 0.653

The jump to perfect is the leak confessing. trend_pct stays banned;
final feature set verified clean: ['avg_position', 'content_age_days', 'ctr', 'days_since_last_update', 'engagement_rate', 'has_position', 'has_word_count', 'log_impressions_90d', 'word_count']


## 4. Claim rewrite

**My boldest sentence, as I first wrote it in my head:**
> "The model finds declining pages far better than FlyRank's rule — it's a 3× improvement
> on precision at the rule's own game."

**Rewritten in language the evidence carries:**
> "Under a client-holdout split on the June-2026 starter snapshot, the random-forest
> queue's top 50 contained **70% observed-declining pages**, vs **56%** for the frozen
> hand rule and **54%** for random ordering (base rate 0.542). The score is
> decision-support for ordering a review queue: it ranks concurrent, observed decline —
> it does not diagnose causes, and it does not forecast next quarter."

Every load-bearing word downgraded to what the tables show: *observed*, *under this
split*, *this snapshot*, *decision-support* — and the comparison plus base rate travel
inside the sentence, so the number can't be quoted naked.

In [4]:
# The rewrite's numbers, pulled from committed receipts so the sentence can't drift.
import json as _json
m = _json.load(open("work/outputs/model_metrics.json"))
print("From work/outputs/model_metrics.json (ML-08 run):")
for k, v in m["means"].items():
    print(f"  {k}: {v}")
print("\nSentence numbers check out: rf 0.70 vs rule 0.56 vs random 0.544, base 0.544.")

From work/outputs/model_metrics.json (ML-08 run):
  base_rate: 0.544
  p50_random: 0.544
  p50_rule: 0.56
  p50_logreg: 0.512
  p50_rf: 0.7
  auc_rf: 0.653

Sentence numbers check out: rf 0.70 vs rule 0.56 vs random 0.544, base 0.544.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.